# Lumis SDK — one-notebook synthetic playground

**Run all cells from top to bottom.** Everything here is synthetic: no live backends, API keys, paid models, Kubernetes, Docker or automatic fixes.

You will:
1. Load a real Lumis YAML project and inspect its configuration.
2. Feed synthetic telemetry through the **real Prometheus, Loki, Tempo and Prefect connectors**.
3. Display the full NetworkX-backed estate, a smaller incident graph and a candidate path.
4. Compare evidence-backed deterministic triage with a scripted tool-using investigator.
5. Watch progress and try a genuine asynchronous text-streaming protocol with a scripted model.
6. Test missing/degraded evidence and save a report plus a separate human-review record.

The fixtures are next to this notebook in [synthetic/](synthetic/). You can edit `lumis.yaml`, `incident.json` and `telemetry.json`, then restart the kernel and run again. Each full run uses a fresh temporary directory.

**Kernel:** select **Lumis SDK playground**. Setup details are in [the notebook README](README.md). Source metadata may still say `0.1.0rc1`; these examples require the reviewed operational SDK with Loki/Tempo/Prefect support, not an older index artifact.


In [ ]:
import asyncio
import copy
import html
import importlib.metadata
import json
import shutil
import sys
import tempfile
from pathlib import Path

import httpx
from IPython.display import HTML, SVG, display
from pydantic_ai import Agent
from pydantic_ai.messages import ModelResponse, ToolCallPart
from pydantic_ai.models.function import FunctionModel

import lumis_sdk
from lumis_sdk.core import GraphSnapshot, Hypothesis, Incident
from lumis_sdk.graph.render import svg
from lumis_sdk.investigation.agent import PydanticInvestigator
from lumis_sdk.investigation.contracts import HumanResolution
from lumis_sdk.runtime import IncidentStore, YamlProject

SDK_ROOT = Path(lumis_sdk.__file__).resolve().parents[2]
FIXTURES = SDK_ROOT / "docs" / "notebooks" / "synthetic"
assert (FIXTURES / "lumis.yaml").is_file(), "Use the local editable SDK and the playground kernel."
print("Python:", sys.executable)
print("SDK source:", SDK_ROOT)
print("Lumis:", importlib.metadata.version("lumis-sdk"))
print("Pydantic AI:", importlib.metadata.version("pydantic-ai-slim"))

## 1. Synthetic YAML and inputs

Our forecast service depends on feature preparation and a database. A Prefect workflow schedules the work; Tempo spans observe the feature-to-forecast call.

The YAML registers approved observation queries, one terminal deterministic signature and an exact source-file allowlist. The mocked endpoints use `.synthetic.test` and are handled entirely in memory. **Removing the mock client does not turn this into a real deployment.**

The fictional incident window is 12:00–12:10 UTC on 3 October 2026. Invented observations are: **8 retries, 2 error logs, an 1800 ms trace and a 120 s failed workflow**. The notebook's 300 ms comparison baseline is illustrative, not measured production performance.


In [ ]:
workspace = Path(tempfile.mkdtemp(prefix="lumis-playground-"))
shutil.copyfile(FIXTURES / "lumis.yaml", workspace / "lumis.yaml")
shutil.copyfile(FIXTURES / "incident.json", workspace / "incident.json")
approved_source = workspace / "approved-source"
approved_source.mkdir()
shutil.copyfile(FIXTURES / "features.py", approved_source / "features.py")

telemetry = json.loads((FIXTURES / "telemetry.json").read_text())
project = YamlProject.from_file(workspace / "lumis.yaml")  # local validation only
event = Incident.model_validate_json((workspace / "incident.json").read_text())
assert project.config.sources.tempo.max_trace_reads == 5
print("Run artifacts:", workspace)
print((workspace / "lumis.yaml").read_text())

## 2. Synthetic backends, real connectors

`httpx.MockTransport` replaces the network only. Lumis still builds the requests, validates responses, normalizes facts, enforces bounds and records provenance.

The fixture handler supports the fixed read APIs used here. Any unexpected host/path is rejected. Prefect's filter POSTs are read-only. Every injected HTTP client is closed by its caller. Discovery reads topology; investigation retrieves evidence independently.

Keep this cell if you edit the YAML: simply pointing a mocked project at localhost is not a live integration test.


In [ ]:
http_calls = []


def fixture_transport(*, missing_logs=False, trace_unavailable=False):
    data = copy.deepcopy(telemetry)
    if missing_logs:
        data["loki"]["data"]["result"] = []

    def handle(request):
        approved_hosts = {
            "prometheus.synthetic.test",
            "loki.synthetic.test",
            "tempo.synthetic.test",
            "prefect.synthetic.test",
        }
        assert request.url.host in approved_hosts, "No real network endpoint is approved here."
        http_calls.append(
            {"method": request.method, "host": request.url.host, "path": request.url.path}
        )
        path = request.url.path
        if request.method == "GET" and path == "/api/v1/query":
            result = copy.deepcopy(data["prometheus"])
            result["data"]["result"][0] = float(request.url.params["time"])
            return httpx.Response(200, json=result)
        if request.method == "GET" and path == "/loki/api/v1/query_range":
            return httpx.Response(200, json=data["loki"])
        if request.method == "GET" and path == "/api/search":
            return (
                httpx.Response(503)
                if trace_unavailable
                else httpx.Response(200, json=data["tempo_search"])
            )
        if request.method == "GET" and path.startswith("/api/traces/"):
            assert path.rsplit("/", 1)[-1] == data["tempo_search"]["traces"][0]["traceID"]
            return httpx.Response(200, json=data["tempo_trace"])
        if request.method == "POST" and path == "/api/flow_runs/filter":
            body = json.loads(request.content)
            assert body["flows"]["name"]["any_"] == ["daily-forecast"]
            return httpx.Response(200, json=data["prefect_flow_runs"])
        if request.method == "POST" and path == "/api/task_runs/filter":
            body = json.loads(request.content)
            assert body["sort"] == "EXPECTED_START_TIME_ASC"
            assert body["flow_runs"]["id"]["any_"] == [data["prefect_flow_runs"][0]["id"]]
            return httpx.Response(200, json=data["prefect_task_runs"])
        raise AssertionError(f"Unsupported synthetic request: {request.method} {path}")

    return httpx.MockTransport(handle)


def table(rows):
    if not rows:
        display(HTML("<p>No rows.</p>"))
        return
    keys = list(rows[0])
    header = "".join(f"<th>{html.escape(key)}</th>" for key in keys)
    body = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(row.get(key, '')))}</td>" for key in keys) + "</tr>"
        for row in rows
    )
    display(
        HTML(
            "<style>.lumis-table{border-collapse:collapse;font:14px system-ui;width:100%}"
            ".lumis-table td,.lumis-table th{padding:9px;border-bottom:1px solid "
            "#ddd;text-align:left}"
            ".lumis-table th{background:#f1f5f9}</style>"
            f"<table class='lumis-table'><thead><tr>{header}</tr></thead><tbody>"
            f"{body}</tbody></table>"
        )
    )


async with httpx.AsyncClient(transport=fixture_transport(), timeout=5) as client:
    prepared = await project.prepare(client=client, at=event.ended_at)
assert prepared.discovery.complete
table(
    [
        {
            "source": source.name,
            "status": source.status,
            "entities": source.entities,
            "relationships": source.relationships,
        }
        for source in prepared.discovery.sources
    ]
)


def show_svg(data, description):
    # Validate with IPython, then embed inline so narrow editor/HTML views fit the graph.
    SVG(data)
    accessible = data.replace(
        "<svg ",
        '<svg role="img" aria-label="'
        + html.escape(description, quote=True)
        + '" style="max-width:100%;height:auto;display:block;" ',
        1,
    )
    display(HTML('<div style="width:100%;overflow:auto;">' + accessible + "</div>"))

## 3. See the graphs

The SDK graph is a NetworkX **directed multigraph**: distinct relationship types are retained. Arrows show declared/observed dependencies or containment, **not proven causality**. Prefect run/task IDs and service-call relationships came from synthetic discovery, not hard-coded graph placeholders.

The SDK's SVG renderer needs no Graphviz or matplotlib. We render the full estate first, then a two-hop incident neighborhood. Hover a node to see its canonical ID.


In [ ]:
estate = prepared.graph.snapshot()
networkx_graph = prepared.graph.to_networkx()
assert networkx_graph.is_multigraph() and networkx_graph.is_directed()
print(f"Full estate: {len(estate.entities)} entities / {len(estate.relationships)} relationships")
estate_svg = svg(estate)
(workspace / "estate.svg").write_text(estate_svg)
show_svg(estate_svg, "Full operational estate: nine entities and ten relationships")

In [ ]:
target = event.affected_entities[0]
neighborhood = prepared.graph.dependencies_within(target, hops=2, max_entities=20)
show_svg(svg(neighborhood), "Two-hop incident dependency neighborhood")
print("Upstream dependencies:", prepared.graph.upstream_of(target, hops=3))
print("Downstream dependants:", prepared.graph.downstream_of(target, hops=2))
table(
    [
        {
            "from": edge.source,
            "relationship": edge.kind,
            "to": edge.target,
            "provenance": ", ".join(edge.provenance),
        }
        for edge in neighborhood.relationships
    ]
)

## 4. A deterministic signature can be enough

The fictional operator approved a signature requiring **two distinct observable facts**: at least five database retries and at least one database error record. It also has a falsifier and explicit affected-entity scope.

Expected: `match → deterministic → supported_diagnosis`, **zero model requests**, and human review still required. This demonstrates an approved rule, not automatic rule learning or confirmed causality.


In [ ]:
async with httpx.AsyncClient(transport=fixture_transport(), timeout=5) as client:
    deterministic_report = await prepared.handle_incident(event, client=client)
assert deterministic_report.route == "deterministic"
assert deterministic_report.metrics.model_requests == 0
assert deterministic_report.findings[0].status == "match"
assert deterministic_report.requires_human_review
table(
    [
        {
            "route": deterministic_report.route,
            "conclusion": deterministic_report.conclusion,
            "model requests": deterministic_report.metrics.model_requests,
            "evidence queries": deterministic_report.metrics.evidence_queries,
            "truth state": deterministic_report.truth_state,
        }
    ]
)
table(
    [
        {
            "query": fact.query_id,
            "source": fact.source,
            "value": fact.value,
            "quality": fact.quality,
            "time": fact.observed_at.isoformat(),
        }
        for fact in deterministic_report.context.evidence
    ]
)

## 5. Missing evidence sends us to the investigator

Now the same backend returns **no matching logs**. Lumis must not invent a zero or assume absence proves a condition false. The deterministic signature becomes `unknown`.

We use the real `PydanticInvestigator` with a **scripted FunctionModel**. The script requests the tool catalog, graph, trace durations, matching spans, workflow runtime, task states and the approved source file. It returns a competing, falsifiable candidate; Lumis computes its support from collected facts.

The expanding panel below shows model-request progress generated by this notebook wrapper. It is not private reasoning, a native Lumis event API or proof of model intelligence.


In [ ]:
candidate = Hypothesis(
    id="slow-workflow-path",
    statement=(
        "The observed slow workflow and long forecast trace are consistent with a slow "
        "execution path; the specific root cause needs review."
    ),
    causal_path=("workflow:demo:daily-forecast", "service:demo:features", "service:demo:forecast"),
    predictions=(
        {
            "entity_id": "service:demo:forecast",
            "key": "trace_duration_ms",
            "operator": "gt",
            "value": 1000,
        },
        {
            "entity_id": "workflow:demo:daily-forecast",
            "key": "run_duration_ms",
            "operator": "gt",
            "value": 100000,
        },
    ),
    falsifiers=(
        {
            "entity_id": "service:demo:forecast",
            "key": "trace_duration_ms",
            "operator": "lt",
            "value": 500,
        },
    ),
    evidence_needed=("forecast-traces", "flow-duration"),
)


def make_investigator():
    steps = [
        {"operation": "catalog"},
        {"operation": "graph", "target": target},
        {"operation": "evidence", "query_id": "forecast-traces"},
        {"operation": "evidence", "query_id": "forecast-spans"},
        {"operation": "evidence", "query_id": "flow-duration"},
        {"operation": "evidence", "query_id": "flow-tasks"},
        {"operation": "code.read", "target": "demo-source", "path": "features.py"},
    ]
    progress = []
    panel = display(HTML("<p>Waiting for scripted investigator...</p>"), display_id=True)

    def scripted_model(messages, info):
        index = len(progress)
        if index < len(steps):
            request = steps[index]
            label = request["operation"] + " " + request.get("query_id", request.get("path", ""))
            progress.append(label.strip())
            panel.update(
                HTML(
                    "<strong>Scripted investigation progress</strong><ol>"
                    + "".join(f"<li>{html.escape(step)}</li>" for step in progress)
                    + "</ol>"
                )
            )
            return ModelResponse(
                parts=[
                    ToolCallPart(
                        "inspect", {"request": request}, tool_call_id=f"inspect-{index + 1}"
                    )
                ]
            )
        progress.append("Return candidate; SDK evaluates evidence")
        panel.update(
            HTML(
                "<strong>Scripted investigation progress</strong><ol>"
                + "".join(f"<li>{html.escape(step)}</li>" for step in progress)
                + "</ol>"
            )
        )
        return ModelResponse(
            parts=[
                ToolCallPart(
                    info.output_tools[0].name,
                    {
                        "hypotheses": [candidate.model_dump(mode="json")],
                        "suggestions": [
                            {
                                "hypothesis_id": candidate.id,
                                "description": "Review per-item feature access "
                                "and consider batching "
                                "only after separate measurement; no change has been applied.",
                            }
                        ],
                        "unresolved_questions": [
                            "Do retries or per-item feature reads account for the slow path?"
                        ],
                    },
                    tool_call_id="candidate-report",
                )
            ]
        )

    return PydanticInvestigator(FunctionModel(scripted_model)), progress

In [ ]:
investigator, agent_progress = make_investigator()
agent_event = event.model_copy(update={"id": "synthetic-agent-investigation"})
async with httpx.AsyncClient(transport=fixture_transport(missing_logs=True), timeout=5) as client:
    agent_report = await prepared.handle_incident(
        agent_event, client=client, investigator=investigator
    )
assert agent_report.stop_reason == "agent_completed"
assert agent_report.route == "agent"
assert agent_report.findings[0].status == "unknown"
assert agent_report.assessments[0].state == "supported"
assert agent_report.metrics.model_requests == 8
assert agent_report.truth_state == "unconfirmed_hypothesis"
assert "do-not-export" not in agent_report.model_dump_json()
table(
    [
        {
            "route": agent_report.route,
            "conclusion": agent_report.conclusion,
            "candidate": assessment.hypothesis.id,
            "state": assessment.state,
            "supporting facts": len(assessment.supporting_evidence_ids),
        }
        for assessment in agent_report.assessments
    ]
)
table(
    [
        {
            "query": fact.query_id,
            "source": fact.source,
            "value / excerpt": str(fact.value)[:150],
            "quality": fact.quality,
        }
        for fact in agent_report.context.evidence
    ]
)
table(
    [
        {"operation": receipt.operation, "status": receipt.status, "purpose": receipt.purpose}
        for receipt in agent_report.receipts
    ]
)
print("Usage:", agent_report.metrics.model_dump())
print("Suggestion:", agent_report.suggestions[0].description)

## 6. Inspect the proposed path and simple analysis

This graph shows only the candidate's entities and the relationships already present in the estate. It does **not** turn the candidate into a proven root cause.

The tiny latency chart compares our invented 300 ms baseline with the 1800 ms trace. Six times slower is descriptive arithmetic, not a calibrated diagnosis or probability.


In [ ]:
path_ids = set(candidate.causal_path)
candidate_graph = GraphSnapshot(
    entities=tuple(entity for entity in estate.entities if entity.id in path_ids),
    relationships=tuple(
        edge for edge in estate.relationships if edge.source in path_ids and edge.target in path_ids
    ),
)
show_svg(
    svg(candidate_graph), "Unconfirmed candidate path from workflow through features to forecast"
)

observed_ms = next(
    fact.value for fact in agent_report.context.evidence if fact.query_id == "forecast-traces"
)
baseline_ms = 300
ratio = observed_ms / baseline_ms
latency_svg = (
    '<svg xmlns="http://www.w3.org/2000/svg" width="680" height="180" viewBox="0 0 680 180">'
    '<rect width="680" height="180" fill="#f8fafc"/>'
    '<text x="20" y="28" font-size="16" font-family="sans-serif">'
    "Illustrative latency comparison (ms)</text>"
    f'<text x="20" y="70" font-family="sans-serif">Baseline: {baseline_ms}</text>'
    f'<rect x="190" y="48" width="{baseline_ms / 4}" height="28" fill="#94a3b8"/>'
    f'<text x="20" y="120" font-family="sans-serif">Observed: {observed_ms:g}</text>'
    f'<rect x="190" y="98" width="{observed_ms / 4}" height="28" fill="#2563eb"/>'
    '<text x="20" y="160" font-family="sans-serif">Synthetic ratio: '
    f"{ratio:g}x — not causal proof</text></svg>"
)
show_svg(
    latency_svg, "Synthetic latency: baseline 300 milliseconds versus observed 1800 milliseconds"
)
assert ratio == 6

## 7. Small model-streaming protocol demo

Lumis's current `handle_incident()` returns a final validated `IncidentReport`; it does not expose a native token-stream API.

Below, a separate Pydantic AI `Agent.run_stream()` streams a short **scripted** explanation based on the report. The async chunks are real streaming protocol events, but their content is predetermined: no remote model, tool investigation or assessment is performed here. This text never becomes evidence.

Reference: [Pydantic AI text streaming](https://pydantic.dev/docs/ai/core-concepts/output/#streaming-text). To evaluate a real model later, use explicitly approved provider settings; do not simply treat streamed prose as a validated report.


In [ ]:
async def synthetic_text_stream(messages, info):
    chunks = [
        "Synthetic report: ",
        "the forecast trace lasted 1800 ms. ",
        "The workflow accumulated 120 seconds of runtime. ",
        "These support the candidate checks, not a confirmed root cause. ",
        "Human review is required.",
    ]
    for chunk in chunks:
        await asyncio.sleep(0.03)
        yield chunk


stream_agent = Agent(FunctionModel(stream_function=synthetic_text_stream), output_type=str)
stream_chunks = []
stream_panel = display(HTML("<p>Streaming scripted summary...</p>"), display_id=True)
async with stream_agent.run_stream("Explain this synthetic report briefly.") as streamed:
    async for chunk in streamed.stream_text(delta=True, debounce_by=None):
        stream_chunks.append(chunk)
        stream_panel.update(HTML("<p>" + html.escape("".join(stream_chunks)) + "</p>"))
stream_text = "".join(stream_chunks)
assert len(stream_chunks) >= 2
assert "Human review is required." in stream_text
print("Received streamed chunks:", len(stream_chunks))

## 8. Two quick safety checks

**Capped logs:** set the source limit to two. The backend returns two records, so the count is `degraded`, not enough to certify the terminal rule.

**Unavailable trace query:** reuse the already prepared graph, then return HTTP 503 for trace search. The investigator still collects other evidence, but its trace-based candidate stays `unresolved`. This simulates an evidence failure after preparation. An enabled topology-discovery failure would instead block a fresh `prepare()`.

These are SDK behavior checks, not benchmark/model-quality evaluations.


In [ ]:
from lumis_sdk.runtime.project import OperationalProject

capped_config = project.config.model_dump()
capped_config["sources"]["loki"]["max_results"] = 2
capped_project = YamlProject(OperationalProject.model_validate(capped_config), workspace)
capped_event = event.model_copy(update={"id": "synthetic-capped-logs"})
async with httpx.AsyncClient(transport=fixture_transport(), timeout=5) as client:
    capped_prepared = await capped_project.prepare(client=client, at=event.ended_at)
    capped_report = await capped_prepared.handle_incident(capped_event, client=client)
assert capped_report.route == "human"
assert capped_report.findings[0].status == "unknown"
assert (
    next(fact for fact in capped_report.context.evidence if fact.query_id == "db-errors").quality
    == "degraded"
)

failure_investigator, _ = make_investigator()
failure_event = event.model_copy(update={"id": "synthetic-unavailable-traces"})
async with httpx.AsyncClient(
    transport=fixture_transport(missing_logs=True, trace_unavailable=True), timeout=5
) as client:
    failure_report = await prepared.handle_incident(
        failure_event, client=client, investigator=failure_investigator
    )
assert failure_report.stop_reason == "agent_completed"
assert failure_report.assessments[0].state == "unresolved"
assert any(receipt.status == "error" for receipt in failure_report.receipts)
table(
    [
        {
            "case": "Known signature",
            "route": deterministic_report.route,
            "state": deterministic_report.findings[0].status,
            "model requests": 0,
        },
        {
            "case": "Missing logs / investigator",
            "route": agent_report.route,
            "state": agent_report.assessments[0].state,
            "model requests": agent_report.metrics.model_requests,
        },
        {
            "case": "Capped logs",
            "route": capped_report.route,
            "state": capped_report.findings[0].status,
            "model requests": 0,
        },
        {
            "case": "Unavailable traces",
            "route": failure_report.route,
            "state": failure_report.assessments[0].state,
            "model requests": failure_report.metrics.model_requests,
        },
    ]
)

## 9. Save the result and a separate human record

The incident report, evidence and tool receipts are immutable audit records. A human resolution is a separate attestation; it does not silently confirm the hypothesis or apply a patch.

We record an **inconclusive synthetic review**, with no changes applied. No generated code, remediation, Git operation or sandbox execution is part of this notebook.


In [ ]:
store = IncidentStore(workspace / "incidents.sqlite")
for report in (deterministic_report, agent_report, capped_report, failure_report):
    # Allows rerunning this cell; changing a report under the same ID is not permitted.
    existing = store.get(report.context.incident.id)
    if existing is None:
        store.save(report)
    else:
        assert existing == report

resolution = HumanResolution(
    id="synthetic-human-review",
    incident_id=agent_event.id,
    reviewer="notebook-operator",
    recorded_at=event.ended_at,
    summary="Synthetic tool and evidence flow reviewed; no operational diagnosis confirmed.",
    applied_change="No change applied.",
    outcome="inconclusive",
)
if not store.resolutions(agent_event.id):
    store.record_resolution(resolution)
assert store.get(agent_event.id) == agent_report
assert store.get(agent_event.id).truth_state == "unconfirmed_hypothesis"
assert store.resolutions(agent_event.id) == (resolution,)
assert (approved_source / "features.py").read_text() == (FIXTURES / "features.py").read_text()
(workspace / "agent-report.json").write_text(agent_report.model_dump_json(indent=2))
(workspace / "candidate-path.svg").write_text(svg(candidate_graph))
print("Saved:", workspace / "incidents.sqlite")
print("Saved:", workspace / "agent-report.json")
print("Saved:", workspace / "estate.svg")
print("Source file unchanged. All notebook assertions passed.")

## Try changing one thing

- Change Prometheus's `"8"` to `"1"` in `synthetic/telemetry.json`: the retry falsifier becomes true and the known signature should no longer match. Some deliberately fixed assertions will then fail; inspect why.
- Remove Loki's stream rows: deterministic triage should be unknown, not a fabricated zero.
- Change the trace duration to 400 ms **and adjust span timestamps consistently**: the scripted candidate's trace falsifier should contradict it.
- Reduce `budget.max_queries`: inspect which evidence was not acquired.
- Add an entity/edge to the YAML, prepare again and compare the graph.
- Replace fixture-backed transport with approved real configuration only when doing a separate integration test. Never pass hidden injected-fault labels into Lumis.

Restore the fixtures and restart/run all to get the original expected results.

**What this notebook does not prove:** live provider quality, production readiness, complete lineage, causal certainty or automatic recovery. The model is scripted, the telemetry is invented, and all fixes remain manual. For real source setup see [telemetry connectors](../telemetry-connectors.md).
